In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from gurobipy import gurobipy as gp
from gurobipy import Model, GRB, quicksum

#Excel-Datei auswerten
excel_file = Path(r"C:\1Masterarbeit\Szenario_2\Inputdaten_1.xlsx")
input_folder = excel_file.parent
df = pd.read_excel(excel_file)

df_products = df.iloc[0:2, 1:5].copy()
df_products.columns = ['db_i', 'b_i', 'e_i', 'v_i']
df_products = df_products.astype(float)

df_scharfe = df.iloc[3:5, [1]].copy()
df_scharfe.columns = ['Wert']
df_scharfe.index = ['E_max', 'W_max']
df_scharfe = df_scharfe.astype(float)

df_unscharfe = df.iloc[6:7, 1:3].copy()
df_unscharfe.columns = ['Untergrenze', 'Obergrenze']
df_unscharfe.index = ['BV_max']
df_unscharfe = df_unscharfe.astype(float)

n_produkte = len(df_products)
db, b, e, v = (
    df_products['db_i'].values,
    df_products['b_i'].values,
    df_products['e_i'].values,
    df_products['v_i'].values,
)
E_max = df_scharfe.loc['E_max', 'Wert']
W_max = df_scharfe.loc['W_max', 'Wert']
BV_unter = df_unscharfe.loc['BV_max', 'Untergrenze']
BV_ober = df_unscharfe.loc['BV_max', 'Obergrenze']
d_BV = BV_ober - BV_unter

data = {
    'Produkt': ['1', '2'],
    'db': [df_products.iloc[0]['db_i'], df_products.iloc[1]['db_i']],
    'b (BV)': [df_products.iloc[0]['b_i'], df_products.iloc[1]['b_i']],
    'e (THG)': [df_products.iloc[0]['e_i'], df_products.iloc[1]['e_i']],
    'v (Wasser)': [df_products.iloc[0]['v_i'], df_products.iloc[1]['v_i']],
}

df_table = pd.DataFrame(data)

#Optimierungsproblem
def solve_fuzzy_scenario(scenario_name, only_BV=False):
    
    def calc_G_toleranz(BV_limit):
        m = Model("G_toleranz")
        m.Params.OutputFlag = 0 

        x = [m.addVar(name=f"x{i+1}", lb=0) for i in range(n_produkte)]

        # Zielfunktion
        m.setObjective(quicksum(db[i] * x[i] for i in range(n_produkte)), GRB.MAXIMIZE)

        # NB BV
        m.addConstr(quicksum(b[i] * x[i] for i in range(n_produkte)) <= BV_limit, "BV_limit")

        # NB Umwelt
        if not only_BV:
            m.addConstr(quicksum(e[i] * x[i] for i in range(n_produkte)) <= E_max, "E_scharf")
            m.addConstr(quicksum(v[i] * x[i] for i in range(n_produkte)) <= W_max, "W_scharf")

        m.optimize()
        return m.objVal if m.status == GRB.OPTIMAL else 0.0

    G_U = calc_G_toleranz(BV_unter)
    G_O = calc_G_toleranz(BV_ober)

    # Modell mit lambda
    m = Model(f"Fuzzy_{scenario_name}")
    m.Params.OutputFlag = 0

    x = [m.addVar(name=f"x{i+1}", lb=0) for i in range(n_produkte)]
    lam = m.addVar(name="lambda", lb=0, ub=1, vtype=GRB.CONTINUOUS)

    # ZF -> max lambda
    m.setObjective(lam, GRB.MAXIMIZE)

    zf_sum = quicksum(db[i] * x[i] for i in range(n_produkte))
    bv_sum = quicksum(b[i] * x[i] for i in range(n_produkte))

    # unscharfe ZF
    m.addConstr(lam * (G_O - G_U) - zf_sum <= -G_U, "ZF_unscharf")

    # unscahrfe BV
    m.addConstr(lam * d_BV + bv_sum <= BV_ober, "BV_unscharf")

    # NB Umwelt
    e_sum = quicksum(e[i] * x[i] for i in range(n_produkte))
    v_sum = quicksum(v[i] * x[i] for i in range(n_produkte))
    if not only_BV:
        m.addConstr(e_sum <= E_max, "E_scharf")
        m.addConstr(v_sum <= W_max, "W_scharf")

    m.optimize()

    if m.status != GRB.OPTIMAL:
        lam_opt   = 0.0
        x_opt     = [0.0] * n_produkte
        G_opt, BV_opt, E_opt, W_opt = 0.0, 0.0, 0.0, 0.0
    else:
        lam_opt   = lam.X
        x_opt     = [x[i].X for i in range(n_produkte)]
        zb_values = [db[i]*x[i].X for i in range(n_produkte)]
        bv_values = [b[i]*x[i].X  for i in range(n_produkte)]
        e_values  = [e[i]*x[i].X  for i in range(n_produkte)]
        v_values  = [v[i]*x[i].X  for i in range(n_produkte)]

        G_opt   = sum(zb_values)
        BV_opt  = sum(bv_values)
        E_opt   = sum(e_values)
        W_opt   = sum(v_values)


    return {
        'scenario': scenario_name,
        'lambda': lam_opt,
        'G': G_opt,
        'x': x_opt,
        'BV': BV_opt,
        'E': E_opt,
        'W': W_opt,
        'G_U': G_U,
        'G_O': G_O,
        'model': m,
    }
    

results_2A = solve_fuzzy_scenario("2-A", only_BV=True)
results_2H = solve_fuzzy_scenario("2-H", only_BV=False)


df_2A = pd.DataFrame([{
    'Szenario': results_2A['scenario'],
    'lambda': results_2A['lambda'],
    'G': results_2A['G'],
    'G_U': results_2A['G_U'],
    'G_O': results_2A['G_O'],
    'x1': results_2A['x'][0],
    'x2': results_2A['x'][1],
    'BV': results_2A['BV'],
    'BV_erlaubt': BV_unter + d_BV,
    'E': results_2A['E'],
    'W': results_2A['W']
}])

df_2H = pd.DataFrame([{
    'Szenario': results_2H['scenario'],
    'lambda': results_2H['lambda'],
    'G': results_2H['G'],
    'G_U': results_2H['G_U'],
    'G_O': results_2H['G_O'],
    'x1': results_2H['x'][0],
    'x2': results_2H['x'][1],
    'BV': results_2H['BV'],
    'BV_erlaubt': BV_unter + d_BV,
    'E': results_2H['E'],
    'E_max': E_max,
    'W': results_2H['W'],
    'W_max': W_max
}])

# SA
def run_sensitivity_analysis(scenario_name, only_BV):
    
    deltas = [-0.10, -0.05, 0.00, 0.05, 0.10]
    results = []
    
    for delta_low in deltas:
        for delta_high in deltas:
            BV_unter_new = BV_unter * (1 + delta_low)
            BV_ober_new = BV_ober * (1 + delta_high)
            d_BV_new = BV_ober_new - BV_unter_new

            name_suffix = f"{scenario_name}_U{delta_low:+.0%}_O{delta_high:+.0%}"

            def calc_G_toleranz(BV_limit):
                m = Model("G_toleranz_sens")
                m.Params.OutputFlag = 0
                x = [m.addVar(name=f"x{i+1}", lb=0) for i in range(n_produkte)]
                m.setObjective(quicksum(db[i] * x[i] for i in range(n_produkte)), GRB.MAXIMIZE)
                m.addConstr(quicksum(b[i] * x[i] for i in range(n_produkte)) <= BV_limit, "BV_limit")
                if not only_BV:
                    m.addConstr(quicksum(e[i] * x[i] for i in range(n_produkte)) <= E_max, "E_scharf")
                    m.addConstr(quicksum(v[i] * x[i] for i in range(n_produkte)) <= W_max, "W_scharf")
                m.optimize()
                return m.objVal if m.status == GRB.OPTIMAL else 0.0

            G_U_new = calc_G_toleranz(BV_unter_new)
            G_O_new = calc_G_toleranz(BV_ober_new)

            m = Model(f"Fuzzy_{name_suffix}")
            m.Params.OutputFlag = 0
            x = [m.addVar(name=f"x{i+1}", lb=0) for i in range(n_produkte)]
            lam = m.addVar(name="lambda", lb=0, ub=1, vtype=GRB.CONTINUOUS)
            m.setObjective(lam, GRB.MAXIMIZE)

            zf_sum = quicksum(db[i] * x[i] for i in range(n_produkte))
            bv_sum = quicksum(b[i] * x[i] for i in range(n_produkte))
            m.addConstr(lam * (G_O_new - G_U_new) - zf_sum <= -G_U_new, "ZF_unscharf")
            m.addConstr(lam * d_BV_new + bv_sum <= BV_ober_new, "BV_unscharf")

            if not only_BV:
                e_sum = quicksum(e[i] * x[i] for i in range(n_produkte))
                v_sum = quicksum(v[i] * x[i] for i in range(n_produkte))
                m.addConstr(e_sum <= E_max, "E_scharf")
                m.addConstr(v_sum <= W_max, "W_scharf")

            m.optimize()

            if m.status != GRB.OPTIMAL:
                lam_opt, x_opt = 0.0, [0.0] * n_produkte
                G_opt = E_opt = W_opt = BV_opt = 0.0
                BV_erlaubt = BV_ober_new

            else:
                lam_opt = lam.X
                x_opt = [x[i].X for i in range(n_produkte)]
                G_opt = sum(db[i]*x[i].X for i in range(n_produkte))
                BV_opt = sum(b[i]*x[i].X for i in range(n_produkte))
                E_opt = sum(e[i]*x[i].X for i in range(n_produkte)) if not only_BV else 0.0
                W_opt = sum(v[i]*x[i].X for i in range(n_produkte)) if not only_BV else 0.0
                BV_erlaubt = BV_unter_new + d_BV_new
            row = {
                'delta_low': delta_low,
                'delta_high': delta_high,
                'BV_unter_new': BV_unter_new,
                'BV_ober_new': BV_ober_new,
                'd_BV': d_BV_new,
                'lambda': lam_opt,
                'G': G_opt,
                'G_U': G_U_new,
                'G_O': G_O_new,
                'x1': x_opt[0],
                'x2': x_opt[1],
                'BV': BV_opt,
                'BV_erlaubt': BV_erlaubt,
                'E': E_opt,
                'E_max': E_max if not only_BV else np.nan,
                'W': W_opt,
                'W_max': W_max if not only_BV else np.nan,
            }
            results.append(row)

    return pd.DataFrame(results)

df_sens_2A = run_sensitivity_analysis("2-A", only_BV=True)
df_sens_2H = run_sensitivity_analysis("2-H", only_BV=False)

#Excel Export
output_file = input_folder / "Ergebnisse.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_2A.to_excel(writer, sheet_name='Lösung_2A', index=False)
    df_2H.to_excel(writer, sheet_name='Lösung_2H', index=False)
    df_sens_2A.to_excel(writer, sheet_name='SA_2A', index=False)
    df_sens_2H.to_excel(writer, sheet_name='SA_2H', index=False)


